### Get started with `siluxApi.phntwk`

- define structure
- build Phntwk from structure
    - require wavelength defined previously
- initialize helper(`PhntwkTree`)

In [ ]:
from siluxApi.phntwk.core import StageStructure, PhntwkTree
import numpy as np
import pandas as pd

def define_mzi_structure():
    return StageStructure(
        name='mzi',
        type="cascade",
        components=[
            (2, 2, "dc"),
            StageStructure(
                name="delaypair",
                type="parallel",
                components=[
                    (1, 1, "delay"),
                    (1, 1, "delay"),
                ]
            ),
            (2, 2, "dc"),
        ]
    )

wavelength = np.linspace(1.2, 1.3, 101)

def get_helper():
    structure = define_mzi_structure()
    mzi = structure.build(wavelength)
    helper = PhntwkTree(mzi)
    return mzi, helper



In [ ]:
mzi, helper = get_helper()
helper.print_tree()

### prepare matrix data

- dc
- waveguide

In [ ]:
from siluxApi.phntwk.passive import dc
from siluxApi.phntwk.core.transfer_matrix import stage


def get_dc(wavelength: np.ndarray):
    kappa = 0.05 
    theta = 0.41
    couple_length = 2
    df = pd.DataFrame({"wavelength": wavelength})
    df["couple_length"] = couple_length
    df["kappa"] = kappa
    df["theta"] = theta
    df["couple_loss"] = 0.
    df["fanout_loss"] = 0.
    df = dc.props_2_trans_matrix(dc.params_2_props(df))
    return stage.to_matrix(df)

get_dc(wavelength)


In [ ]:
from siluxApi.phntwk.passive import waveguide as wg
def get_waveguide(wavelength, length):
    neff = 1.65
    df = pd.DataFrame({
        "wavelength": wavelength,
    })
    df["neff"] = neff
    df["length"] = length
    df["loss"] = 0
    df = wg.props_2_trans_matrix(wg.params_2_props(df))
    return stage.to_matrix(df)

get_waveguide(wavelength, 25)
    
    


### add data to Phntwk

In [ ]:
helper.print_tree()

In [ ]:
helper.set_node_matrix("dc", 0, get_dc(wavelength))
helper.set_node_matrix("dc", 1, get_dc(wavelength))
helper.set_node_matrix("delay", 0, get_waveguide(wavelength, 25))
helper.set_node_matrix("delay", 1, get_waveguide(wavelength, 30))
helper.get_node_matrix("mzi")

In [ ]:
import matplotlib.pyplot as plt

df = stage.to_dataframe(mzi)

plt.plot(df["wavelength"], df["t_1_1"], label="through_coef")
plt.plot(df["wavelength"], df["t_2_1"], label="cross_coef")
plt.plot(df["wavelength"], np.abs(df["t_1_1"])**2, label="through_transmission")
plt.plot(df["wavelength"], np.abs(df["t_2_1"])**2, label="cross_transmission")
plt.plot(df["wavelength"], np.abs(df["t_1_1"])**2+np.abs(df["t_2_1"])**2, label="total_transmission")
plt.legend()
